![image_1777347486383.png](./image_1777347486383.png "image_1777347486383.png")

In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col, initcap

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"
# Read races data from bronze layer
bronze_races_df = spark.table(bronze_table)

In [0]:
# Apply transformations
silver_races_df = (
    bronze_races_df
    # Drop URL column
    .drop("url")
    # Remove duplicates
    .dropDuplicates(["season","round"])
    # Rename columns to snake_case and meaningful names using dictionary
    .withColumnsRenamed(
        {
            "raceName": "race_name",
            "circuitId": "circuit_id",
            "date": "race_date",
        }
    )
    # Transform race_name to Title Case
    .withColumn("race_name", initcap(col("race_name")))
)

In [0]:
# Write transformed data to silver layer
(silver_races_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table))

print(f"✓ Races data successfully written to {catalog_name}.{silver_schema}.races")

In [0]:
# silver_races_df.count()

In [0]:
# silver_races_df.limit(10).display()